# Task A KIM batch #1+2 - KIM-I and the gated zero-knowledge control

**Goal.** Test the only KIM component that survived the full-data ablation
(Chen et al., 2018): **local inference** ("I"). This is not a BiLSTM KIM
replacement. It adds a small, gated residual at the **last decoder block** of
the existing A160 xattn model:

```
h_i = h_i + gamma * G([h_i; ctx_i; h_i - ctx_i; h_i * ctx_i; k_i])
ctx_i = sum_j A_ij h_v,j
k_i   = sum_j A_ij R_ij
```

`A` is the last cross-attention map, `R_ij` is the five-dimensional WordNet
relation vector (synonymy, antonymy, hypernymy, hyponymy, co-hyponymy), and
`gamma` is a zero-initialized gate. A fresh KIM-I model is exactly A160 at
step 0, so the run measures whether the model chooses to open the branch.

**Arms (3 seeds each, split-seed 42, video-cluster split):**

| arm | decoder | knowledge | what it tests |
|---|---|---|---|
| `A160r` | A160 | none | same-session control |
| `KIMI` | A160 + last-block local inference | WordNet 5D | KIM-I |
| `KIM0` (gated) | same local-inference block | all zeros | zero-knowledge control |

The controls run first. `KIMI` runs second. `KIM0` runs third, in the same
session, only if the KIM-I gate below passes. Everything else is the recorded
A160 recipe (`dropout .3`, `wd .01`, `head-lr 1e-3`, `xattn-lr 1e-4`,
`backbone-lr 2e-5`, `unfreeze 3`, `layers 3`, shared encoder, no pooled-v,
`pool-init .02`, `160/256`, 15 epochs, patience 5).

**Success bar.** Adopt KIM-I only if all of the following hold against the
same-session `A160r` control:

1. 3-seed mean best macro-F1 gain >= **+0.5 pt**;
2. 3-seed ensemble gain >= **+0.5 pt**;
3. paired video-cluster bootstrap 95% CI for the ensemble delta excludes zero;
4. no severe language regression (no language worse by more than 2 pt);
5. the error count falls in the **polite-stereotype** (`yes|no`) and
   **slur-no-claim** (`no|yes_implicit`, `no|yes_explicit`) quadrants.

If KIM-I fails, the notebook skips the zero-knowledge LI control and stops the
line. If it passes, the same session continues with `KIM0` seeds 42-44. This is
why batch 1 and batch 2 are combined here: Kaggle does not carry `weights/`
between sessions, and re-uploading probability dumps is error-prone. KIM-A is
not implemented and remains deferred.

**Checklist before Run All:**

1. Accelerator: **GPU T4 x2**
2. Internet: **On** (git, pip, model, NLTK corpora)
3. Add Input -> Datasets -> `amnpham/lgbtqia`
4. Optional: if you already ran batch 1, attach its `weights/results_small.zip`
   as a dataset. The import cell copies the probability dumps and histories into
   `weights/`, and the six batch-1 runs are skipped.

**Runtime:** ~6-9 h for six batch-1 runs plus up to three gated `KIM0` runs and
cache prewarm. The notebook is resume-safe: a run with history and probabilities
already in `weights/` is skipped, so re-running in the same session continues.
A fresh session with no imported batch-1 output starts from scratch.

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/AmnO-O/Try.git'
PIN_COMMIT = '70c3b660fb5bc6b53a35e48c65cdd000b9ac84ea'


def sh(cmd):
    print(f'$ {cmd}', flush=True)
    subprocess.run(cmd, shell=True, check=True)


# Re-run safe: after the first run the cwd is already <working>/Try.
if os.path.basename(os.path.abspath('.')) != 'Try':
    if not os.path.isdir('Try'):
        sh(f'git clone {REPO_URL}')
    os.chdir('Try')
if PIN_COMMIT:
    sh('git fetch origin main')
    sh(f'git checkout {PIN_COMMIT}')
sh('git log -1 --format="%H %cd %s" --date=iso')
sh('pip install -q -r requirements.txt')
print('cwd:', os.getcwd(), flush=True)

## 1. Data, NLTK, and the split-42 validation frame

NLTK's WordNet is the KIM knowledge source. The five relation features are built
from WordNet synsets; Italian and Dutch lemmas go through Open Multilingual
WordNet (`omw-1.4` plus `omw-2.0`; the WordNet reader still opens `omw-1.4`
for provenance, so both are required). The corpora are downloaded once to `nltk_data/` in the
working directory.

The validation frame is fixed exactly as in the A160 batches: `val-fraction .1`,
`split-seed 42`, video-cluster split. Expected `738` rows, `276` positives, and
`56` video clusters (EN 276, IT 221, NL 241).

In [ ]:
import glob
import os
import sys
import time

hits = sorted(glob.glob('/kaggle/input/**/StereoQueerEval_*_training.tsv', recursive=True))
assert hits, 'StereoQueerEval TSVs not found - attach the dataset (Add Input > amnpham/lgbtqia)'
os.environ['LGBT_DATA_DIR'] = os.path.dirname(hits[0])
os.environ['LGBT_WEIGHTS_DIR'] = os.path.abspath('weights')
os.environ['NLTK_DATA'] = os.path.abspath('nltk_data')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.makedirs('weights', exist_ok=True)
os.makedirs(os.environ['NLTK_DATA'], exist_ok=True)
print('data dir:', os.environ['LGBT_DATA_DIR'], flush=True)
print('nltk dir:', os.environ['NLTK_DATA'], flush=True)

import nltk

for pkg in ('wordnet', 'omw-1.4', 'omw-2.0'):
    ok = nltk.download(pkg, download_dir=os.environ['NLTK_DATA'], quiet=True)
    print(f'nltk {pkg}: {ok}', flush=True)
    assert ok, f'could not download NLTK package {pkg!r}'

from nltk.corpus import wordnet as wn

assert wn.synsets('dog'), 'WordNet is not readable'
assert wn.synsets('cane', lang='ita'), 'Italian OMW is not readable'
assert wn.synsets('hond', lang='nld'), 'Dutch OMW is not readable'
print('NLTK WordNet + OMW ready', flush=True)

import torch

print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), flush=True)
assert torch.cuda.is_available(), 'No GPU - set Settings > Accelerator > GPU T4 x2, then Run All'
print('gpu:', torch.cuda.get_device_name(0), flush=True)

sys.path.insert(0, os.path.abspath('src'))
import numpy as np
import pandas as pd
import data as data_mod
import knowledge as knowledge_mod
import model_xattn
from model import load_tokenizer
import train_xattn as tx

dev = tx.resolve_device('cuda')
tokenizer = load_tokenizer(model_xattn.DEFAULT_BACKBONE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.sep_token
print('tokenizer fast:', getattr(tokenizer, 'is_fast', False),
      '| sep:', tx.sep_for(tokenizer), flush=True)

df_all = data_mod.load_frame(languages=['EN', 'IT', 'NL'])
df_train, df_val = data_mod.split_by_video(df_all, 0.1, 42)
labs = df_val['st_y'].astype(int).to_numpy()
assert len(labs) == 738, f'unexpected val size: {len(labs)}'
assert int(labs.sum()) == 276, f'unexpected positive count: {int(labs.sum())}'
assert df_val['yt_title'].nunique() == 56, df_val['yt_title'].nunique()

np.save('weights/val_labels.npy', labs)
np.save('weights/val_langs.npy', df_val['lang'].to_numpy(dtype=str))
np.save('weights/val_videos.npy', df_val['yt_title'].to_numpy(dtype=str))
np.save('weights/val_ids.npy', df_val['StereoQueerEval_id'].to_numpy(dtype=str))
quads = (df_val['stereotype'].astype(str) + '|' + df_val['hate_speech'].astype(str)).to_numpy(dtype=str)
np.save('weights/val_quads.npy', quads)

print('val:', len(df_val), 'rows | pos', int(labs.sum()), '| clusters',
      df_val['yt_title'].nunique(), flush=True)
print('  languages', dict(pd.Series(df_val['lang']).value_counts()), flush=True)
print('  quadrants', dict(pd.Series(quads).value_counts()), flush=True)
print('train:', len(df_train), 'rows | languages',
      dict(pd.Series(df_train['lang']).value_counts()), flush=True)

## 2. Prewarm the WordNet cache

The cache is keyed by `(language, comment text, video text)` and stores
word-level relation matrices. `PairedDataset` expands each matrix to token-level
with the tokenizer's offset mapping, so this prewarm uses the **same dataset
class and the same text construction** as training.

Chunking matters on Kaggle: each chunk is saved, so a session that dies halfway
does not lose the WordNet work. Six training runs then share one cache. The
diagnostic below is the sparsity check: KIM's relation matrix is expected to be
sparse, and a near-zero rate would make the arm uninformative regardless of the
gate.

In [ ]:
# Reuse an earlier batch-1 run instead of retraining it. This searches the
# attached Kaggle datasets for the small results zip and for loose dumps, so a
# fresh session can start at the KIM0 control without redoing A160r/KIMI.
import shutil
import zipfile

BATCH1_IMPORT_DIR = os.environ.get('BATCH1_IMPORT_DIR', '').strip()
import_roots = [r for r in (BATCH1_IMPORT_DIR, '/kaggle/input') if r and os.path.isdir(r)]
imported = []
for root in import_roots:
    for zp in sorted(glob.glob(os.path.join(root, '**', 'results_small.zip'), recursive=True)):
        with zipfile.ZipFile(zp) as zf:
            for name in zf.namelist():
                base = os.path.basename(name)
                if not base.endswith(('.csv', '.npy', '.json')):
                    continue
                dst = os.path.join('weights', base)
                if os.path.exists(dst):
                    continue
                with zf.open(name) as src, open(dst, 'wb') as fh:
                    shutil.copyfileobj(src, fh)
                imported.append(base)
    for pat in ('A160r_s*_val_probs.npy', 'A160r_s*_history.csv',
                'KIMI_s*_val_probs.npy', 'KIMI_s*_history.csv',
                'KIM0_s*_val_probs.npy', 'KIM0_s*_history.csv'):
        for src in sorted(glob.glob(os.path.join(root, '**', pat), recursive=True)):
            dst = os.path.join('weights', os.path.basename(src))
            if os.path.exists(dst):
                continue
            shutil.copy2(src, dst)
            imported.append(os.path.basename(src))
print(f'batch-1 import roots: {import_roots} | imported {len(imported)} file(s)', flush=True)
if imported:
    names = sorted(set(imported))
    print('  ' + ', '.join(names[:10]) + (' ...' if len(names) > 10 else ''), flush=True)

KIM_CACHE = os.path.abspath(os.path.join('weights', 'kim_word_cache.pkl'))
KIM_DIM = 5
KIM_EDGES = 8
CHUNK = 300

print('KIM cache:', KIM_CACHE, flush=True)
cache = knowledge_mod.KnowledgeCache(KIM_CACHE, max_hypernym_edges=KIM_EDGES)
print('existing cached pairs:', len(cache), flush=True)

nonzero = 0
total = 0
rows_with = 0
seen = 0
feature_nonzero = np.zeros(KIM_DIM, dtype=np.int64)
t0 = time.time()

for start in range(0, len(df_all), CHUNK):
    chunk = df_all.iloc[start:start + CHUNK].reset_index(drop=True)
    ds = tx.PairedDataset(
        chunk, tokenizer,
        comment_max_len=160, video_max_len=256,
        with_knowledge=True, knowledge_cache=cache,
        knowledge_dim=KIM_DIM, zero_knowledge=False,
    )
    for k in ds.knowledge:
        nonzero += int(np.count_nonzero(k))
        total += int(k.size)
        rows_with += int(np.any(k))
        feature_nonzero += np.count_nonzero(k, axis=(0, 1)).astype(np.int64)
    seen += len(ds)
    del ds
    cache.save()
    rate = nonzero / max(total, 1)
    print(f'  {seen:>5}/{len(df_all)} rows | cache {len(cache):>5} pairs | '
          f'nonzero {rate:.4%} | {(time.time() - t0) / 60:.1f} min', flush=True)

assert seen == len(df_all), (seen, len(df_all))
assert rows_with > 0, 'no row received any WordNet relation'
rate = nonzero / max(total, 1)
print(f'\ncache pairs: {len(cache)} | rows with a relation: {rows_with}/{seen} '
      f'({rows_with / seen:.1%}) | nonzero entries: {nonzero}/{total} ({rate:.3%})', flush=True)
for name, count in zip(knowledge_mod.FEATURE_NAMES, feature_nonzero.tolist()):
    print(f'  {name:<12} {count:>9}', flush=True)
if rate < 0.001:
    print('WARNING: relation matrix is extremely sparse; KIM-I may be a no-op.', flush=True)
print('KIM cache prewarm complete', flush=True)

## 3. Run list and helpers

The architecture and recipe are identical to the recorded A160r control. The
only change in `KIMI` is `--kim-local` plus the WordNet cache. The KIM gate is
logged per epoch as `kim_gate`; a run whose gate stays at zero is the baseline
plus unused parameters, not a knowledge result.

In [ ]:
BASE = dict(
    dropout=0.3, weight_decay=0.01, head_lr=1e-3, xattn_lr=1e-4,
    backbone_lr=2e-5, unfreeze_layers=3, layers=3, shared_encoder=True,
    pooled_v=False, freeze_epochs=0, pool_init_scale=0.02,
    max_len=160, video_max_len=256, no_amp=False,
    backbone=model_xattn.DEFAULT_BACKBONE,
)

RUNS = []
for s in (42, 43, 44):
    RUNS.append(dict(BASE, tag=f'A160r_s{s}', seed=s, split_seed=42, kim_local=False))
for s in (42, 43, 44):
    RUNS.append(dict(BASE, tag=f'KIMI_s{s}', seed=s, split_seed=42, kim_local=True))

# batch-6 recorded A160r values: continuity check, not the primary pairing.
RECORDED = {'A160r_s42': 0.7476, 'A160r_s43': 0.7396, 'A160r_s44': 0.7276}


def train_one(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    probs = f"weights/{r['tag']}_val_probs.npy"
    if os.path.exists(hist) and os.path.exists(probs):
        print(f"-- skip (history + probabilities exist): {r['tag']}", flush=True)
        return True
    if os.path.exists(hist) and os.path.exists(out):
        print(f"-- history + checkpoint exist; probabilities will be dumped: {r['tag']}",
              flush=True)
        return True

    kim_args = ''
    if r['kim_local']:
        kim_args = (f"--kim-local --kim-cache {KIM_CACHE} "
                    f"--kim-knowledge-dim {KIM_DIM} --kim-max-hypernym-edges {KIM_EDGES} ")
    cmd = (
        'python src/train_xattn.py --languages EN IT NL --epochs 15 --patience 5 --batch-size 32 '
        f"--backbone {r['backbone']} "
        f"--max-len {r['max_len']} --video-max-len {r['video_max_len']} "
        f"--dropout {r['dropout']} --weight-decay {r['weight_decay']} "
        f"--head-lr {r['head_lr']} --xattn-lr {r['xattn_lr']} --backbone-lr {r['backbone_lr']} "
        f"--unfreeze-layers {r['unfreeze_layers']} --layers {r['layers']} "
        f"--pool-init-scale {r['pool_init_scale']} --freeze-epochs {r['freeze_epochs']} "
        f"--context-dropout 0 "
        f"{'--pooled-v' if r['pooled_v'] else '--no-pooled-v'} "
        f"{'--shared-encoder ' if r['shared_encoder'] else ''}"
        f"{kim_args}"
        f"--split-seed {r['split_seed']} --seed {r['seed']} "
        f"--out {out} --history {hist} --probs-out {probs}"
    )
    if r['no_amp']:
        cmd += ' --no-amp'
    print(f'$ {cmd}', flush=True)
    t0 = time.time()
    rc = subprocess.run(cmd, shell=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'}).returncode
    print(f"== {r['tag']}: exit {rc} after {(time.time() - t0) / 60:.1f} min", flush=True)
    if rc != 0 or not os.path.exists(hist):
        print(f'!! {r["tag"]} failed', flush=True)
        return False

    h = pd.read_csv(hist)
    bi = h['val_macro_f1'].idxmax()
    best = float(h.loc[bi, 'val_macro_f1'])
    extra = ''
    rec = RECORDED.get(r['tag'])
    if rec is not None:
        extra = f' | batch-6 same-seed {rec:.4f} (delta {best - rec:+.4f})'
    gate_txt = ''
    if 'kim_gate' in h.columns:
        gate_best = float(h.loc[bi, 'kim_gate'])
        gate_last = float(h['kim_gate'].iloc[-1])
        gate_max = float(h['kim_gate'].max())
        gate_txt = (f' | kim_gate best {gate_best:+.5f} last {gate_last:+.5f} '
                    f'max {gate_max:+.5f}')
        if r['kim_local'] and gate_max < 1e-4:
            gate_txt += ' !! gate stayed closed'
    print(f"   {r['tag']}: best epoch {int(h.loc[bi, 'epoch'])} | "
          f"val_macro_f1 {best:.4f} | acc {h.loc[bi, 'val_acc']:.4f}{extra}{gate_txt} | "
          f"{len(h)} epochs", flush=True)
    return True


def dump_probs(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    p = f"weights/{r['tag']}_val_probs.npy"
    if os.path.exists(p) and np.load(p).shape == (len(labs), 2):
        print(f'-- skip probs: {r["tag"]}', flush=True)
        return
    if not (os.path.exists(hist) and os.path.exists(out)):
        print(f'-- no checkpoint/history for {r["tag"]}', flush=True)
        return
    net = model_xattn.build_model(
        r['backbone'], unfreeze_layers=r['unfreeze_layers'],
        heads=4, layers=r['layers'], dropout=r['dropout'], ffn_mult=4,
        use_pooled_v=r['pooled_v'], separate_encoders=not r['shared_encoder'],
        keep_frozen_dropout=False, pool_init_scale=r['pool_init_scale'],
        num_classes=2, kim_local=r['kim_local'], knowledge_dim=KIM_DIM,
    ).to(dev)
    net.load_state_dict(torch.load(out, map_location=dev))
    net.eval()
    loader = tx.make_loader(
        df_val, tokenizer, 32, r['max_len'], r['video_max_len'],
        with_knowledge=r['kim_local'],
        knowledge_cache=cache if r['kim_local'] else None,
        knowledge_dim=KIM_DIM,
    )
    o = tx.collect(net, loader, dev, use_amp=True, with_knowledge=r['kim_local'])
    np.save(p, o['probs'])
    m = tx.metrics_from(o['preds'], o['labels'])
    print(f"   {r['tag']}: reload macro_f1 {m['macro_f1']:.4f} acc {m['acc']:.4f}", flush=True)
    del net
    torch.cuda.empty_cache()


def zip_small():
    import zipfile
    small = [f for f in sorted(os.listdir('weights'))
             if f.endswith(('.csv', '.npy', '.json'))]
    with zipfile.ZipFile('weights/results_small.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in small:
            zf.write(os.path.join('weights', f), f)
    print(f'wrote weights/results_small.zip ({len(small)} files)', flush=True)


print('planned:', len(RUNS), 'runs |', [r['tag'] for r in RUNS], flush=True)

## 4. Train

Controls first, then KIM-I. Resume-safe. The first `KIMI_s42` run is the canary:
if it fails, the loop stops instead of burning the remaining GPU budget. Small
results are zipped after every run so a dropped session still leaves the
histories and probability dumps.

In [ ]:
t0 = time.time()
for i, r in enumerate(RUNS, 1):
    print(f'=== [{i}/{len(RUNS)}] {r["tag"]} (elapsed {(time.time() - t0) / 60:.0f} min) ===',
          flush=True)
    ok = False
    try:
        ok = train_one(r)
    except Exception as e:                           # noqa: BLE001
        print(f'!! train_one({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
    if ok:
        try:
            dump_probs(r)
        except Exception as e:                       # noqa: BLE001
            print(f'!! dump_probs({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
    try:
        zip_small()
    except Exception as e:                           # noqa: BLE001
        print(f'!! zip_small raised: {type(e).__name__}: {e}', flush=True)
    if not ok:
        print('!! stopping this batch after the first failed run; fix and re-run to resume',
              flush=True)
        break
print(f'batch finished in {(time.time() - t0) / 3600:.2f} h', flush=True)

## 5. Per-seed, ensemble, language, and quadrant summary

`mean best` is the mean of each run's best validation macro-F1 from its history.
`reload` is the metric recomputed from the saved best checkpoint and raw
probabilities. They should match; a mismatch means the checkpoint/probability
path is stale. The ensemble is the mean of the three seed probability matrices.

The quadrant table separates the two failure modes that matter for Task A:

- **polite stereotype** = gold `yes|no` (stereotype, no hate cue), reported as FN;
- **slur, no claim** = gold `no|yes_implicit` + `no|yes_explicit` (hate cue, no
  stereotype claim), reported as FP combined.

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score

labs = np.load('weights/val_labels.npy')
langs = np.load('weights/val_langs.npy', allow_pickle=True).astype(str)
quads = np.load('weights/val_quads.npy', allow_pickle=True).astype(str)

rows = []
probs_by_tag = {}
for r in RUNS:
    hist = f"weights/{r['tag']}_history.csv"
    p = f"weights/{r['tag']}_val_probs.npy"
    if not (os.path.exists(hist) and os.path.exists(p)):
        print(f'missing history/probs: {r["tag"]}', flush=True)
        continue
    h = pd.read_csv(hist)
    bi = h['val_macro_f1'].idxmax()
    probs = np.load(p)
    pred = probs.argmax(1)
    m = tx.metrics_from(pred.tolist(), labs.tolist())
    auc = float(roc_auc_score(labs, probs[:, 1]))
    probs_by_tag[r['tag']] = probs
    row = dict(
        arm=r['tag'].rsplit('_s', 1)[0], tag=r['tag'], seed=r['seed'],
        best_epoch=int(h.loc[bi, 'epoch']), epochs=len(h),
        best_hist_f1=float(h.loc[bi, 'val_macro_f1']),
        reload_f1=float(m['macro_f1']), reload_acc=float(m['acc']),
        auc=auc, pred_pos=float((pred == 1).mean()),
        f1_yes=float(m['f1_yes']), precision_yes=float(m['precision_yes']),
        recall_yes=float(m['recall_yes']),
    )
    if 'kim_gate' in h.columns:
        row['kim_gate_best'] = float(h.loc[bi, 'kim_gate'])
        row['kim_gate_last'] = float(h['kim_gate'].iloc[-1])
        row['kim_gate_max'] = float(h['kim_gate'].max())
    rows.append(row)
    print(f"{r['tag']:<12} ep {int(h.loc[bi, 'epoch']):>2} | "
          f"hist {float(h.loc[bi, 'val_macro_f1']):.4f} | "
          f"reload {m['macro_f1']:.4f} | AUC {auc:.4f} | pos {pred.mean():.3f}", flush=True)

if not rows:
    raise RuntimeError('No completed runs: nothing to summarize. Re-run the training cell.')
summ = pd.DataFrame(rows).sort_values(['arm', 'seed'])
summ.to_csv('weights/taskA_kim_batch1_summary.csv', index=False)
print()
print(summ.to_string(index=False), flush=True)

QUAD_ORDER = ('yes|no', 'yes|yes_implicit', 'yes|yes_explicit',
              'no|no', 'no|yes_implicit', 'no|yes_explicit')


def per_language(probs):
    out = {}
    for lang in sorted(set(langs.tolist())):
        msk = langs == lang
        m = tx.metrics_from(probs[msk].argmax(1).tolist(), labs[msk].tolist())
        out[lang] = float(m['macro_f1'])
    return out


def quadrant_errors(probs):
    pred = probs.argmax(1)
    out = {}
    for q in QUAD_ORDER:
        m = quads == q
        n = int(m.sum())
        if q.startswith('yes'):
            err = int((pred[m] == 0).sum())
        else:
            err = int((pred[m] == 1).sum())
        out[q] = (err, n)
    return out


def target_error_counts(probs):
    qe = quadrant_errors(probs)
    polite = qe['yes|no'][0]
    slur = qe['no|yes_implicit'][0] + qe['no|yes_explicit'][0]
    return polite, slur


arms = {}
arm_rows = []
ens_by_arm = {}
for arm in ('A160r', 'KIMI'):
    tags = [f'{arm}_s{s}' for s in (42, 43, 44)]
    tags = [t for t in tags if t in probs_by_tag]
    if len(tags) != 3:
        print(f'{arm}: {len(tags)}/3 seeds; skipped', flush=True)
        continue
    arm_df = summ[summ.arm == arm]
    mean_best = float(arm_df['best_hist_f1'].mean())
    mean_reload = float(arm_df['reload_f1'].mean())
    ens = np.mean([probs_by_tag[t] for t in tags], axis=0)
    em = tx.metrics_from(ens.argmax(1).tolist(), labs.tolist())
    np.save(f'weights/taskA_kim_batch1_{arm}_ensemble_probs.npy', ens)
    ens_by_arm[arm] = ens
    arms[arm] = dict(mean_best=mean_best, mean_reload=mean_reload,
                     ensemble_f1=float(em['macro_f1']), ensemble_acc=float(em['acc']),
                     ensemble_auc=float(roc_auc_score(labs, ens[:, 1])),
                     pred_pos=float((ens.argmax(1) == 1).mean()))
    arm_rows.append(dict(arm=arm, mean_best=mean_best, mean_reload=mean_reload,
                         ensemble_f1=arms[arm]['ensemble_f1'],
                         ensemble_acc=arms[arm]['ensemble_acc'],
                         ensemble_auc=arms[arm]['ensemble_auc'],
                         pred_pos=arms[arm]['pred_pos']))
    print(f"\n{arm}: mean best {mean_best:.4f} | mean reload {mean_reload:.4f} | "
          f"ensemble {em['macro_f1']:.4f} | AUC {arms[arm]['ensemble_auc']:.4f} | "
          f"pos {arms[arm]['pred_pos']:.3f}", flush=True)
    print('  per-language ensemble:', per_language(ens), flush=True)
    print('  quadrants:', quadrant_errors(ens), flush=True)

arm_summ = pd.DataFrame(arm_rows)
arm_summ.to_csv('weights/taskA_kim_batch1_arms.csv', index=False)
if len(arm_summ):
    print()
    print(arm_summ.to_string(index=False), flush=True)

if 'A160r' in arms and 'KIMI' in arms:
    dm = arms['KIMI']['mean_best'] - arms['A160r']['mean_best']
    de = arms['KIMI']['ensemble_f1'] - arms['A160r']['ensemble_f1']
    print(f'\nKIMI - A160r: mean best {dm:+.4f} | ensemble {de:+.4f}', flush=True)
    pi, si = target_error_counts(ens_by_arm['KIMI'])
    pb, sb = target_error_counts(ens_by_arm['A160r'])
    print(f'  polite-stereotype errors: A160r {pb} -> KIMI {pi} ({pi - pb:+d})', flush=True)
    print(f'  slur-no-claim errors:     A160r {sb} -> KIMI {si} ({si - sb:+d})', flush=True)

zip_small()
print('summary written', flush=True)

## 6. Paired video-cluster bootstrap

The 738 validation comments are nested in 56 videos. Resampling rows i.i.d.
understates uncertainty, so each bootstrap draw resamples **whole videos with
replacement** and keeps every comment from a selected video. The same resampled
rows are used for both arms, which makes the delta paired.

Reported below: the ensemble delta, its 95% percentile CI, and a two-sided
bootstrap-tail p-value. Also reported: the mean of the three same-seed deltas,
and each seed's paired delta. A positive mean with a CI that includes zero is not
a result at this validation size.

In [ ]:
import json

from sklearn.metrics import f1_score

videos = np.load('weights/val_videos.npy', allow_pickle=True).astype(str)
clusters = np.unique(videos)
groups = [np.flatnonzero(videos == c) for c in clusters]
print(f'{len(clusters)} video clusters | {len(labs)} rows', flush=True)


def macro_f1_binary(y, pred):
    return float(f1_score(y, pred, labels=[0, 1], average='macro', zero_division=0))


def paired_cluster_bootstrap(y, probs_a, probs_b, n_boot=5000, seed=12345):
    rng = np.random.default_rng(seed)
    obs_a = macro_f1_binary(y, probs_a.argmax(1))
    obs_b = macro_f1_binary(y, probs_b.argmax(1))
    diffs = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        pick = rng.integers(0, len(groups), size=len(groups))
        idx = np.concatenate([groups[j] for j in pick])
        diffs[i] = (macro_f1_binary(y[idx], probs_a[idx].argmax(1))
                    - macro_f1_binary(y[idx], probs_b[idx].argmax(1)))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    p = 2.0 * min(float((diffs <= 0).mean()), float((diffs >= 0).mean()))
    return dict(observed=obs_a - obs_b, a=obs_a, b=obs_b,
                bootstrap_mean=float(diffs.mean()),
                ci_low=float(lo), ci_high=float(hi),
                p_value=float(min(1.0, p)), n_boot=n_boot,
                n_clusters=len(groups))


def mean_seed_bootstrap(y, probs_a_list, probs_b_list, n_boot=5000, seed=12345):
    rng = np.random.default_rng(seed)
    obs = float(np.mean([
        macro_f1_binary(y, a.argmax(1)) - macro_f1_binary(y, b.argmax(1))
        for a, b in zip(probs_a_list, probs_b_list)
    ]))
    diffs = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        pick = rng.integers(0, len(groups), size=len(groups))
        idx = np.concatenate([groups[j] for j in pick])
        diffs[i] = float(np.mean([
            macro_f1_binary(y[idx], a[idx].argmax(1))
            - macro_f1_binary(y[idx], b[idx].argmax(1))
            for a, b in zip(probs_a_list, probs_b_list)
        ]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    p = 2.0 * min(float((diffs <= 0).mean()), float((diffs >= 0).mean()))
    return dict(observed=obs, bootstrap_mean=float(diffs.mean()),
                ci_low=float(lo), ci_high=float(hi),
                p_value=float(min(1.0, p)), n_boot=n_boot,
                n_clusters=len(groups))


boot = {}
seed_list = (42, 43, 44)
if 'A160r' in ens_by_arm and 'KIMI' in ens_by_arm:
    boot['ensemble'] = paired_cluster_bootstrap(
        labs, ens_by_arm['KIMI'], ens_by_arm['A160r'], n_boot=5000, seed=12345)
    boot['seed_mean'] = mean_seed_bootstrap(
        labs,
        [probs_by_tag[f'KIMI_s{s}'] for s in seed_list],
        [probs_by_tag[f'A160r_s{s}'] for s in seed_list],
        n_boot=5000, seed=12346)
    boot['per_seed'] = {}
    for s in seed_list:
        boot['per_seed'][s] = paired_cluster_bootstrap(
            labs, probs_by_tag[f'KIMI_s{s}'], probs_by_tag[f'A160r_s{s}'],
            n_boot=5000, seed=12347 + s)
    print('\nensemble KIMI - A160r:', json.dumps(boot['ensemble'], indent=2), flush=True)
    print('mean of per-seed deltas:', json.dumps(boot['seed_mean'], indent=2), flush=True)
    for s in seed_list:
        print(f'  seed {s}:', json.dumps(boot['per_seed'][s]), flush=True)
    with open('weights/taskA_kim_batch1_bootstrap.json', 'w', encoding='utf-8') as fh:
        json.dump(boot, fh, indent=2)

# Decision rule. Main criteria are the mean/ensemble delta and the paired CI.
# The quadrant criterion is a small-n diagnostic: it is reported separately and
# only enters the final PASS as a directional check.
if 'A160r' in arms and 'KIMI' in arms:
    mean_delta = arms['KIMI']['mean_best'] - arms['A160r']['mean_best']
    ens_delta = arms['KIMI']['ensemble_f1'] - arms['A160r']['ensemble_f1']
    ci_low = boot['ensemble']['ci_low']
    lang_base = per_language(ens_by_arm['A160r'])
    lang_kim = per_language(ens_by_arm['KIMI'])
    lang_delta = {k: lang_kim[k] - lang_base[k] for k in lang_base}
    min_lang_delta = min(lang_delta.values())
    polite_delta = target_error_counts(ens_by_arm['KIMI'])[0] - target_error_counts(ens_by_arm['A160r'])[0]
    slur_delta = target_error_counts(ens_by_arm['KIMI'])[1] - target_error_counts(ens_by_arm['A160r'])[1]

    main_pass = (mean_delta >= 0.005) and (ens_delta >= 0.005) and (ci_low > 0.0)
    lang_pass = min_lang_delta >= -0.02
    quad_pass = (polite_delta < 0) and (slur_delta < 0)
    passed = main_pass and lang_pass and quad_pass

    print('\n--- DECISION ---', flush=True)
    print(f'mean best delta   {mean_delta:+.4f} (need >= +0.005): {mean_delta >= 0.005}', flush=True)
    print(f'ensemble delta    {ens_delta:+.4f} (need >= +0.005): {ens_delta >= 0.005}', flush=True)
    print(f'bootstrap CI low  {ci_low:+.4f} (need > 0): {ci_low > 0.0}', flush=True)
    print(f'language deltas   {lang_delta} (need min >= -0.020): {lang_pass}', flush=True)
    print(f'quadrant deltas   polite {polite_delta:+d} | slur {slur_delta:+d} '
          f'(need both < 0): {quad_pass}', flush=True)
    print('KIM-I PASSES the pre-registered bar' if passed else
          'KIM-I DOES NOT clear the bar', flush=True)
    if passed:
        print('Next: run batch 2 zero-knowledge LI control. Stage KIM-A only if the '
              'knowledge-specific gain survives that control.', flush=True)
    else:
        print('Do not run KIM-A or the LI control. Record the negative result and move on.',
              flush=True)
    decision = dict(mean_delta=float(mean_delta), ensemble_delta=float(ens_delta),
                    ci_low=float(ci_low), language_deltas={k: float(v) for k, v in lang_delta.items()},
                    polite_delta=int(polite_delta), slur_delta=int(slur_delta),
                    main_pass=bool(main_pass), language_pass=bool(lang_pass),
                    quadrant_pass=bool(quad_pass), passed=bool(passed))
    with open('weights/taskA_kim_batch1_decision.json', 'w', encoding='utf-8') as fh:
        json.dump(decision, fh, indent=2)

zip_small()
print('TASK A KIM BATCH1 COMPLETE', flush=True)

## 7. Batch-1 gate in this session

The bootstrap cell above sets `passed` with the exact pre-registered rule. Reuse
it: if KIM-I failed, `KIM0` is skipped and no GPU time is spent on the control.
The gate is evaluated on the probability dumps in `weights/`, whether they were
produced earlier in this session or imported from an attached batch-1 output.

In [ ]:
PROCEED_KIM0 = bool(globals().get('passed', False))
if 'passed' not in globals():
    print('Batch-1 decision unavailable; KIM0 is skipped. '
          'Run the batch-1 summary and bootstrap cells first.', flush=True)
elif PROCEED_KIM0:
    print('Batch-1 gate PASSED: KIM0 will run in this session.', flush=True)
else:
    print('Batch-1 gate FAILED: KIM0 is skipped; KIM-A is not justified.', flush=True)

## 8. KIM0 - zero-knowledge local-inference control (gated)

`KIM0` builds the exact KIM-I architecture but feeds an all-zero knowledge
matrix. The comparison separates the attention-residual block from the WordNet
features:

- `KIM0 - A160r` = the local-inference block with zero knowledge;
- `KIMI - KIM0` = the five WordNet features with the same architecture.

This cell runs only when `PROCEED_KIM0` is true. It is resume-safe: a run with
history and probabilities in `weights/` is skipped.

In [ ]:
SEEDS = (42, 43, 44)
KIM0_RUNS = [dict(BASE, tag=f'KIM0_s{s}', seed=s, split_seed=42,
                  kim_local=True, zero_knowledge=True,
                  knowledge_dim=KIM_DIM) for s in SEEDS]


def train_kim0(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    probs = f"weights/{r['tag']}_val_probs.npy"
    if os.path.exists(hist) and os.path.exists(probs):
        print(f"-- skip (history + probabilities exist): {r['tag']}", flush=True)
        return True
    if os.path.exists(hist) and os.path.exists(out):
        print(f"-- history + checkpoint exist; probabilities will be dumped: {r['tag']}",
              flush=True)
        return True
    cmd = (
        'python src/train_xattn.py --languages EN IT NL --epochs 15 --patience 5 --batch-size 32 '
        f"--backbone {r['backbone']} "
        f"--max-len {r['max_len']} --video-max-len {r['video_max_len']} "
        f"--dropout {r['dropout']} --weight-decay {r['weight_decay']} "
        f"--head-lr {r['head_lr']} --xattn-lr {r['xattn_lr']} --backbone-lr {r['backbone_lr']} "
        f"--unfreeze-layers {r['unfreeze_layers']} --layers {r['layers']} "
        f"--pool-init-scale {r['pool_init_scale']} --freeze-epochs {r['freeze_epochs']} "
        f"--context-dropout 0 "
        f"{'--pooled-v' if r['pooled_v'] else '--no-pooled-v'} "
        f"{'--shared-encoder ' if r['shared_encoder'] else ''}"
        f"--kim-local --kim-zero-knowledge --kim-knowledge-dim {r['knowledge_dim']} "
        f"--split-seed {r['split_seed']} --seed {r['seed']} "
        f"--out {out} --history {hist} --probs-out {probs}"
    )
    if r['no_amp']:
        cmd += ' --no-amp'
    print(f'$ {cmd}', flush=True)
    t0 = time.time()
    rc = subprocess.run(cmd, shell=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'}).returncode
    print(f"== {r['tag']}: exit {rc} after {(time.time() - t0) / 60:.1f} min", flush=True)
    if rc != 0 or not os.path.exists(hist):
        print(f'!! {r["tag"]} failed', flush=True)
        return False
    h = pd.read_csv(hist)
    bi = h['val_macro_f1'].idxmax()
    gate_txt = ''
    if 'kim_gate' in h.columns:
        gate_txt = (f" | kim_gate best {float(h.loc[bi, 'kim_gate']):+.5f} "
                    f"last {float(h['kim_gate'].iloc[-1]):+.5f} "
                    f"max {float(h['kim_gate'].max()):+.5f}")
    print(f"   {r['tag']}: best epoch {int(h.loc[bi, 'epoch'])} | "
          f"val_macro_f1 {float(h.loc[bi, 'val_macro_f1']):.4f} | "
          f"acc {float(h.loc[bi, 'val_acc']):.4f}{gate_txt} | {len(h)} epochs", flush=True)
    return True


def dump_probs_kim0(r):
    out = f"weights/{r['tag']}.pt"
    p = f"weights/{r['tag']}_val_probs.npy"
    if os.path.exists(p) and np.load(p).shape == (len(labs), 2):
        print(f'-- skip probs: {r["tag"]}', flush=True)
        return
    if not os.path.exists(out):
        print(f'-- no checkpoint for {r["tag"]}', flush=True)
        return
    net = model_xattn.build_model(
        r['backbone'], unfreeze_layers=r['unfreeze_layers'],
        heads=4, layers=r['layers'], dropout=r['dropout'], ffn_mult=4,
        use_pooled_v=r['pooled_v'], separate_encoders=not r['shared_encoder'],
        keep_frozen_dropout=False, pool_init_scale=r['pool_init_scale'],
        num_classes=2, kim_local=True, knowledge_dim=r['knowledge_dim'],
    ).to(dev)
    net.load_state_dict(torch.load(out, map_location=dev))
    net.eval()
    loader = tx.make_loader(
        df_val, tokenizer, 32, r['max_len'], r['video_max_len'],
        with_knowledge=True, knowledge_cache=None,
        knowledge_dim=r['knowledge_dim'], zero_knowledge=True,
    )
    o = tx.collect(net, loader, dev, use_amp=True, with_knowledge=True)
    np.save(p, o['probs'])
    m = tx.metrics_from(o['preds'], o['labels'])
    print(f"   {r['tag']}: reload macro_f1 {m['macro_f1']:.4f} acc {m['acc']:.4f}", flush=True)
    del net
    torch.cuda.empty_cache()


print('KIM0 planned:', len(KIM0_RUNS), 'runs |', [r['tag'] for r in KIM0_RUNS], flush=True)
print('PROCEED_KIM0:', PROCEED_KIM0, flush=True)

## 9. Train KIM0 (only if the gate passed)

If `PROCEED_KIM0` is false, this cell prints the skip reason and runs nothing.
That is the intended outcome when KIM-I did not clear the pre-registered bar.

In [ ]:
if not PROCEED_KIM0:
    print('SKIP KIM0: batch-1 gate failed or decision unavailable. '
          'No zero-knowledge control is run.', flush=True)
else:
    t0 = time.time()
    for i, r in enumerate(KIM0_RUNS, 1):
        print(f'=== [KIM0 {i}/{len(KIM0_RUNS)}] {r["tag"]} '
              f'(elapsed {(time.time() - t0) / 60:.0f} min) ===', flush=True)
        ok = False
        try:
            ok = train_kim0(r)
        except Exception as e:                       # noqa: BLE001
            print(f'!! train_kim0({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
        if ok:
            try:
                dump_probs_kim0(r)
            except Exception as e:                   # noqa: BLE001
                print(f'!! dump_probs_kim0({r["tag"]}) raised: {type(e).__name__}: {e}',
                      flush=True)
        try:
            zip_small()
        except Exception as e:                       # noqa: BLE001
            print(f'!! zip_small raised: {type(e).__name__}: {e}', flush=True)
        if not ok:
            print('!! stopping KIM0 after the first failed run; fix and re-run to resume',
                  flush=True)
            break
    print(f'KIM0 finished in {(time.time() - t0) / 3600:.2f} h', flush=True)

## 10. Combined summary: architecture vs knowledge

Three comparisons matter:

1. `KIM0 - A160r` = value of the local-inference block with **zero** knowledge.
2. `KIMI - KIM0` = value of the five WordNet features with the **same**
   architecture.
3. `KIMI - A160r` = the headline KIM-I result from batch 1.

If `KIM0` captures the whole KIMI gain, KIM-I is an attention-residual effect,
not knowledge injection, and KIM-A is not justified. If `KIMI - KIM0` is
positive with a bootstrap CI above zero, the WordNet path is doing real work.

In [ ]:
if not PROCEED_KIM0:
    print('No combined summary: batch-1 gate did not pass.', flush=True)
else:
    for s in SEEDS:
        p = f'weights/KIM0_s{s}_val_probs.npy'
        if os.path.exists(p):
            probs_by_tag[f'KIM0_s{s}'] = np.load(p)

    def best_history(tag):
        hp = f'weights/{tag}_history.csv'
        if not os.path.exists(hp):
            return float('nan')
        return float(pd.read_csv(hp)['val_macro_f1'].max())

    arm_rows = []
    ens3 = {}
    for arm in ('A160r', 'KIMI', 'KIM0'):
        tags = [f'{arm}_s{s}' for s in SEEDS if f'{arm}_s{s}' in probs_by_tag]
        if len(tags) != 3:
            print(f'{arm}: {len(tags)}/3 seeds; skipped', flush=True)
            continue
        ens = np.mean([probs_by_tag[t] for t in tags], axis=0)
        ens3[arm] = ens
        m = tx.metrics_from(ens.argmax(1).tolist(), labs.tolist())
        hb = [best_history(t) for t in tags]
        valid = [v for v in hb if not np.isnan(v)]
        mean_best = float(np.mean(valid)) if valid else float('nan')
        lang_f1 = {}
        for lang in sorted(set(langs.tolist())):
            msk = langs == lang
            lang_f1[lang] = float(tx.metrics_from(
                ens[msk].argmax(1).tolist(), labs[msk].tolist())['macro_f1'])
        row = dict(arm=arm, mean_best=mean_best,
                   ensemble_f1=float(m['macro_f1']),
                   ensemble_acc=float(m['acc']),
                   ensemble_auc=float(roc_auc_score(labs, ens[:, 1])),
                   pred_pos=float((ens.argmax(1) == 1).mean()))
        row.update({f'f1_{k}': v for k, v in lang_f1.items()})
        arm_rows.append(row)
        np.save(f'weights/taskA_kim_combined_{arm}_ensemble_probs.npy', ens)
        print(f"\n{arm}: mean best {mean_best:.4f} | ensemble {m['macro_f1']:.4f} | "
              f"AUC {row['ensemble_auc']:.4f} | pos {row['pred_pos']:.3f}", flush=True)
        print(f"  per-language: {lang_f1}", flush=True)
        print(f"  quadrants: {quadrant_errors(ens)}", flush=True)

    arm_summ = pd.DataFrame(arm_rows)
    arm_summ.to_csv('weights/taskA_kim_combined_arms.csv', index=False)
    print()
    print(arm_summ.to_string(index=False), flush=True)

    boot2 = {}
    if 'KIM0' in ens3 and 'A160r' in ens3:
        boot2['KIM0_vs_A160r'] = paired_cluster_bootstrap(
            labs, ens3['KIM0'], ens3['A160r'], n_boot=5000, seed=12355)
    if 'KIM0' in ens3 and 'KIMI' in ens3:
        boot2['KIMI_vs_KIM0'] = paired_cluster_bootstrap(
            labs, ens3['KIMI'], ens3['KIM0'], n_boot=5000, seed=12356)
    for key, val in boot2.items():
        print(f'{key}: {json.dumps(val)}', flush=True)
    with open('weights/taskA_kim_combined_bootstrap.json', 'w', encoding='utf-8') as fh:
        json.dump(boot2, fh, indent=2)

    if 'KIM0' in ens3 and 'A160r' in ens3 and 'KIMI' in ens3:
        li_gain = float(arm_summ.loc[arm_summ.arm == 'KIM0', 'ensemble_f1'].iloc[0]
                        - arm_summ.loc[arm_summ.arm == 'A160r', 'ensemble_f1'].iloc[0])
        kg = float(arm_summ.loc[arm_summ.arm == 'KIMI', 'ensemble_f1'].iloc[0]
                   - arm_summ.loc[arm_summ.arm == 'KIM0', 'ensemble_f1'].iloc[0])
        kg_ci_low = float(boot2['KIMI_vs_KIM0']['ci_low'])
        print('\n--- DECISION ---', flush=True)
        print(f'LI architecture (KIM0 - A160r): {li_gain:+.4f}', flush=True)
        print(f'knowledge (KIMI - KIM0):       {kg:+.4f} | CI low {kg_ci_low:+.4f}',
              flush=True)
        if kg >= 0.005 and kg_ci_low > 0:
            verdict = ('Knowledge-specific gain: keep KIM-I. KIM-A is the only '
                       'justified follow-up, but it still needs an implementation.')
        elif kg <= -0.005:
            verdict = ('The zero-knowledge control is better than KIM-I: the WordNet '
                       'features hurt. Stop this line.')
        else:
            verdict = ('No knowledge-specific gain: the local-inference block, not '
                       'WordNet, explains the KIM-I result. Do not implement KIM-A.')
        print(verdict, flush=True)
        with open('weights/taskA_kim_combined_decision.json', 'w', encoding='utf-8') as fh:
            json.dump(dict(li_gain=li_gain, knowledge_gain=kg,
                           knowledge_ci_low=kg_ci_low,
                           proceed_kim0=bool(PROCEED_KIM0)), fh, indent=2)

    zip_small()
    print('TASK A KIM COMBINED COMPLETE', flush=True)